# **Evaluating Lllama 3.1**

In [ ]:
HF_TOKEN = 

In [ ]:
!pip install transformers torch accelerate bitsandbytes sentence-transformers scikit-learn spacy tqdm
!python -m spacy download en_core_web_sm

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 43.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 115.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
import os
import json
import ast
import torch
import spacy
import numpy as np
import time
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import re
import unicodedata
from difflib import SequenceMatcher
from typing import List, Dict, Tuple, Set, Optional

In [ ]:
from huggingface_hub import login
from google.colab import userdata

try:
    # Get the token from Colab secrets
    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        login(token=hf_token)
        print("Hugging Face login successful!")
    else:
        print("HF_TOKEN not found in Colab secrets.")
except Exception as e:
    print(f"An error occurred during Hugging Face login: {e}")

Hugging Face login successful!


In [ ]:
!python -m spacy download en_core_web_sm

  Using cached https://github.com/explosion/spacy-models/releases/download/en_core_web_sm-3.8.0/en_core_web_sm-3.8.0-py3-none-any.whl (12.8 MB)
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [ ]:
!python -m spacy download en_core_web_lg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 5.3 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


# **Llama 3.1 8B**

In [ ]:
import torch
import spacy
import ast
import re
import json
import unicodedata
import numpy as np
from tqdm import tqdm
from typing import List, Dict, Tuple, Optional
from difflib import SequenceMatcher
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sklearn.metrics.pairwise import cosine_similarity

# --- Main Configuration ---
#MODEL_ID = "meta-llama/Meta-Llama-3.1-8B-Instruct"
MODEL_ID = "meta-llama/Meta-Llama-3.1-70B-Instruct"

TRAIN_DATA_PATH = "train.txt"
TEST_DATA_PATH = "test.txt"

# --- Helper Functions ---
def read_line_examples_from_file(data_path: str) -> Tuple[List[str], List[list]]:
    """Reads data from file, each line is: sent####labels"""
    sents, labels = [], []
    with open(data_path, 'r', encoding='UTF-8') as fp:
        for line in fp:
            line = line.strip()
            if line != '' and '####' in line:
                words, tuples_str = line.split('####', 1)
                try:
                    tuples = ast.literal_eval(tuples_str)
                    sents.append(words)
                    labels.append(tuples)
                except (ValueError, SyntaxError):
                    print(f"Skipping malformed line: {line}")
    return sents, labels

# --- Advanced Metrics Calculation Section ---
def _normalize_text(s: str) -> str:
    """ Normalizes unicode, lowercases, strips, removes punctuation, collapses whitespace. """
    if not isinstance(s, str):
        s = str(s)
    s = unicodedata.normalize("NFKC", s)
    s = s.strip().lower()
    s = "".join(ch for ch in s if not unicodedata.category(ch).startswith("P"))
    s = re.sub(r"\s+", " ", s).strip()
    words_to_strip = ['a', 'an', 'the', 'of', 'in', 'on', 'at', 'for', 'to', 'with', 'by']
    words = s.split()
    while words and words[0] in words_to_strip:
        words.pop(0)
    while words and words[-1] in words_to_strip:
        words.pop(-1)
    s = " ".join(words)
    return s

def _opinion_sim(a: str, b: str) -> float:
    """ Uses SequenceMatcher ratio as a lightweight similarity metric for strings. """
    return SequenceMatcher(None, a, b).ratio()

def calculate_metrics(
    all_preds: List[List[List[str]]],
    all_golds: List[List[List[str]]],
    normalize: bool = True,
    dedupe: bool = True,
    opinion_fuzzy_threshold: Optional[float] = None
) -> Dict[str, float]:
    """ Computes micro precision/recall/F1 for quadruple lists with advanced matching options. """
    if len(all_preds) != len(all_golds):
        raise ValueError(f"Length mismatch: preds {len(all_preds)} vs golds {len(all_golds)}")

    total_tp = total_fp = total_fn = 0
    for preds, golds in zip(all_preds, all_golds):
        if normalize:
            g_list = [tuple(_normalize_text(x) for x in label) for label in golds]
            p_list = [tuple(_normalize_text(x) for x in label) for label in preds]
        else:
            g_list = [tuple(str(x) for x in label) for label in golds]
            p_list = [tuple(str(x) for x in label) for label in preds]

        if dedupe:
            g_list = list(dict.fromkeys(g_list))
            p_list = list(dict.fromkeys(p_list))

        matched_pred_idx = set()
        tp = 0
        for g in g_list:
            for pj, p in enumerate(p_list):
                if pj in matched_pred_idx:
                    continue

                non_opinion_match = (g[0] == p[0] and g[1] == p[1] and g[2] == p[2])
                if not non_opinion_match:
                    continue

                opinion_match = False
                if opinion_fuzzy_threshold is None:
                    if g[3] == p[3]:
                        opinion_match = True
                else:
                    sim = _opinion_sim(g[3], p[3])
                    if sim >= opinion_fuzzy_threshold:
                        opinion_match = True

                if opinion_match:
                    tp += 1
                    matched_pred_idx.add(pj)
                    break

        fp = len(p_list) - tp
        fn = len(g_list) - tp
        total_tp += tp
        total_fp += fp
        total_fn += fn

    print(f"\nCalculating scores with advanced metrics...")
    print(f"Total True Positives: {total_tp}, False Positives: {total_fp}, False Negatives: {total_fn}")
    precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0
    recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    return {'precision': precision * 100, 'recall': recall * 100, 'f1': f1 * 100}

class Llama_ASQPExtractor:
    """An extractor class to evaluate Llama 3.1 models via transformers."""
    def __init__(self, model_id: str, token: str = None):
        print(f"Initializing Llama Extractor for model: {model_id}")
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,

        )

        print("Loading tokenizer...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_id, token=token)

        print("Loading model (this may take a while)...")
        self.model = AutoModelForCausalLM.from_pretrained(
            model_id,
            torch_dtype=torch.bfloat16,
            device_map="auto",
            quantization_config=quantization_config,
            token=token,

        )

        self.nlp_sm = spacy.load("en_core_web_sm")
        print("Model and tools initialized successfully!")

    def _create_prompt(self, instruction: str, query: str, dependency_tree: str, few_shot_examples: List[Dict]) -> str:
        """Formats the prompt into the Llama 3.1 Instruct chat format."""
        system_prompt = instruction

        few_shot_text = ""
        if few_shot_examples:
            few_shot_text = "### Examples\nHere are some examples to guide you:\n"
            for example in few_shot_examples:
                few_shot_text += f"\nSentence: {example['sentence']}\nLabels: {str(example['labels'])}\n"
            few_shot_text += "\n### Now, based on instruction and these examples, perform the task for the following sentence:\n"
        else:
            few_shot_text = "### Now, based on the instruction, perform the task for the following sentence:\n"

        user_prompt = f"{few_shot_text}Query: {query}\n"
        user_prompt += f"Dependency Tree: {dependency_tree}\n"
        user_prompt += "LABELS:"

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ]

        formatted_prompt = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        return formatted_prompt

    def _parse_output(self, output_text: str) -> List[list]:
        """Robustly parse the model's string output into a Python list of lists."""
        # Clean up the output text
        output_text = output_text.strip()

        # Find the start of the list
        start = output_text.find('[')
        if start == -1:
            return []

        # Use a stack to find the matching closing bracket
        stack = 0
        end = -1
        for i in range(start, len(output_text)):
            ch = output_text[i]
            if ch == '[':
                stack += 1
            elif ch == ']':
                stack -= 1
                if stack == 0:
                    end = i
                    break
        if end == -1:
            return []

        list_str = output_text[start:end+1]

        # Replace single quotes with double quotes for proper JSON parsing
        # But be careful not to replace quotes inside words (contractions)
        list_str = re.sub(r"(?<![a-zA-Z])'(?![a-zA-Z])", '"', list_str)

        # Try multiple parsing strategies
        try:
            # Strategy 1: Use ast.literal_eval (handles Python syntax)
            parsed_list = ast.literal_eval(list_str)
            if isinstance(parsed_list, list) and all(isinstance(i, (list, tuple)) and len(i) == 4 for i in parsed_list):
                # Convert tuples to lists and ensure all elements are strings
                return [[str(elem) for elem in item] for item in parsed_list]
        except Exception:
            pass

        try:
            # Strategy 2: Use json.loads (handles JSON syntax)
            parsed_list = json.loads(list_str)
            if isinstance(parsed_list, list) and all(isinstance(i, list) and len(i) == 4 for i in parsed_list):
                # Ensure all elements are strings
                return [[str(elem) for elem in item] for item in parsed_list]
        except Exception:
            pass

        # Strategy 3: Manual regex-based parsing as fallback
        try:
            # Extract all quadruples using regex
            quadruple_pattern = r'\[(.*?)\]'
            matches = re.findall(quadruple_pattern, list_str)

            result = []
            for match in matches:
                # Skip the outer list match
                if match == list_str[1:-1]:
                    continue

                # Split by comma but respect quoted strings
                elements = re.findall(r'["\']([^"\']*)["\']|([^,\[\]]+)', match)
                elements = [e[0] if e[0] else e[1].strip() for e in elements if e[0] or e[1].strip()]

                if len(elements) == 4:
                    result.append([str(elem) for elem in elements])

            if result:
                return result
        except Exception:
            pass

        return []

    def extract_quadruples(self, instruction: str, sentence: str, few_shot_examples: List[Dict]) -> List[list]:
        """Gets a prediction from the Llama model for a given sentence."""
        doc = self.nlp_sm(sentence)
        dep_lines = [f"{token.text}-{token.head.text}->{token.dep_}" for token in doc]
        dependency_tree = "\n".join(dep_lines)

        prompt = self._create_prompt(instruction, sentence, dependency_tree, few_shot_examples)

        inputs = self.tokenizer(prompt, return_tensors="pt").to("cuda")

        terminators = [
            self.tokenizer.eos_token_id,
            self.tokenizer.convert_tokens_to_ids("<|eot_id|>")
        ]
        try:
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=512,
                eos_token_id=terminators,
                do_sample=False,
                pad_token_id=self.tokenizer.eos_token_id
            )
            response_text = self.tokenizer.decode(outputs[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)
            parsed = self._parse_output(response_text)
            # Only show debug if parsing failed AND there was actual content
            if not parsed and response_text.strip() and not response_text.strip().startswith('[]'):
                # Check if it looks like it should have been parseable
                if '[' in response_text and ']' in response_text:
                    print(f"⚠ WARNING: Failed to parse output with brackets: \n{response_text[:150]}...")
            return parsed
        except Exception as e:
            print(f" [Model Error]: {e}. Returning empty list.")
            return []

if __name__ == "__main__":
    NUM_FEW_SHOTS = 0
    hf_token = None
    extractor = Llama_ASQPExtractor(model_id=MODEL_ID, token=hf_token)

    print("\nLoading datasets and building the few-shot retriever...")
    train_sents, train_labels = read_line_examples_from_file(TRAIN_DATA_PATH)
    test_sents, test_labels = read_line_examples_from_file(TEST_DATA_PATH)

    if NUM_FEW_SHOTS > 0:
        nlp_lg = spacy.load("en_core_web_lg")
        print("Building few-shot retriever index (this may take a moment)...")
        train_embeddings = np.array([nlp_lg(sent).vector for sent in tqdm(train_sents)])

    def retrieve_few_shots(query_sentence: str, num_examples: int = 1) -> List[Dict]:
        if num_examples <= 0:
            return []
        query_embedding = nlp_lg(query_sentence).vector.reshape(1, -1)
        similarities = cosine_similarity(query_embedding, train_embeddings).flatten()
        top_indices = np.argsort(similarities)[-num_examples:][::-1]
        return [{"sentence": train_sents[idx], "labels": train_labels[idx]} for idx in top_indices]

    instruction_prompt = """
ROLE:
You are an expert ASQP (aspect sentiment quadruple prediction) annotation bot. Your processing must be precise, systematic, and strictly follow all rules.

TASK DEFINITION:
Your sole task is to extract all sentiment quadruples from a user-provided sentence and its dependency tree.
Your response MUST be a valid Python list of lists, with no other text or explanations.
The output format for each quadruple is a list of four strings:
[<Aspect Term>, <Aspect Category>, <Sentiment Polarity>, <Opinion Term>]

ELEMENT DEFINITIONS:
1.  Aspect Term: The specific noun or phrase being reviewed.
    - It MUST be a term from the restaurant domain (e.g., "food", "service", "waiter").
    - Use the exact string "NULL" if the opinion is general or the aspect is implicit.
2.  Aspect Category: The general class for the Aspect Term.
    - You MUST choose from this predefined list only:
        - `ambience general`
        - `drinks prices`
        - `drinks quality`
        - `drinks style_options`
        - `food prices`
        - `food quality`
        - `food style_options`
        - `location general`
        - `restaurant general`
        - `restaurant miscellaneous`
        - `restaurant prices`
        - `service general`
3.  Sentiment: The polarity of the opinion.
    - It MUST be one of three strings: "positive", "negative", or "neutral".
4.  Opinion Term: The exact word(s) from the sentence expressing the sentiment.

EXTRACTION RULES:
1.  Concise Spans: The Opinion Term must be the most concise phrase that still captures the full sentiment. Do not include unnecessary words.
2.  Extract All Opinions: You must extract a separate quadruple for every distinct opinion expressed in the sentence.
3.  Dependency Tree Usage: To ensure accuracy, you must use the dependency tree to find the correct relationship between an opinion and an aspect. Follow the grammatical connections in the tree to link an Opinion Term back to the specific Aspect Term it modifies. This is the most reliable way to create a correct quadruple. If no such link exists, the Aspect Term is "NULL".
4.  Implied Sentiment: Analyze comparative and conditional phrases to determine the implied sentiment.

STEP BY STEP PROCESS:
To ensure accuracy, think step-by-step before generating the final list:
1.  First, identify all opinion-expressing words/phrases in the sentence.
2.  Second, for each opinion, identify its specific aspect phrase. If none exist, that aspect is "NULL".
3.  Third, determine the sentiment polarity (positive, negative, neutral) and the correct aspect category from the provided list.
4.  Finally, construct the list of all quadruples according to all rules above.

FINAL OUTPUT INSTRUCTION:
Your final output MUST ONLY be a valid Python list of lists.
- Do not include explanations, reasoning, or markdown formatting.
- Do not include extra text before or after the list.
- Each inner list must contain exactly 4 strings.
- Use double quotes for strings, not single quotes.
"""

    all_predictions = []
    all_ground_truths = test_labels

    print(f"\n--- Starting Evaluation on {len(test_sents)} Test Samples ---")
    for i, sentence in enumerate(tqdm(test_sents)):
        few_shot_examples = retrieve_few_shots(sentence, num_examples=NUM_FEW_SHOTS) if NUM_FEW_SHOTS > 0 else []
        predicted_quads = extractor.extract_quadruples(instruction_prompt, sentence, few_shot_examples)
        all_predictions.append(predicted_quads)

        if (i + 1) % 10 == 0:
            print(f"\n----- Sample {i+1}/{len(test_sents)} -----")
            print(f" Sentence: {sentence}")
            print(f" Ground Truth: {all_ground_truths[i]}")
            print(f" Prediction: {predicted_quads}")
            print("-" * 25)

    final_metrics = calculate_metrics(all_predictions, all_ground_truths)
    print("\n\n========== EVALUATION COMPLETE ==========")
    print(f" Model: {MODEL_ID}")
    print(f" Few-shots: {NUM_FEW_SHOTS}")
    print(f" Precision: {final_metrics['precision']:.2f}%")
    print(f" Recall: {final_metrics['recall']:.2f}%")
    print(f" F1-Score: {final_metrics['f1']:.2f}%")
    print("=======================================")

    model_name_safe = MODEL_ID.split('/')[-1]
    output_filename = f"Rest15_evaluation_{NUM_FEW_SHOTS}shot_results_{model_name_safe}.json"
    results_to_save = [
        {"sentence": s, "ground_truth": gt, "prediction": p}
        for s, gt, p in zip(test_sents, all_ground_truths, all_predictions)
    ]
    with open(output_filename, "w", encoding="utf-8") as f:
        json.dump(results_to_save, f, indent=4)
    print(f"\nDetailed results saved to '{output_filename}'")

Initializing Llama Extractor for model: meta-llama/Meta-Llama-3.1-70B-Instruct
Loading tokenizer...
Loading model (this may take a while)...


Loading checkpoint shards:   0%|          | 0/30 [00:00<?, ?it/s]

Model and tools initialized successfully!

Loading datasets and building the few-shot retriever...

--- Starting Evaluation on 537 Test Samples ---


  1%|          | 4/537 [00:00<00:38, 13.80it/s]

 [Model Error]: Cannot copy out of meta tensor; no data!. Returning empty list.
 [Model Error]: Cannot copy out of meta tensor; no data!. Returning empty list.
 [Model Error]: Cannot copy out of meta tensor; no data!. Returning empty list.
 [Model Error]: Cannot copy out of meta tensor; no data!. Returning empty list.
 [Model Error]: Cannot copy out of meta tensor; no data!. Returning empty list.


...

 29%|██▉       | 157/537 [00:06<00:17, 22.31it/s]

 [Model Error]: Cannot copy out of meta tensor; no data!. Returning empty list.
 [Model Error]: Cannot copy out of meta tensor; no data!. Returning empty list.
 [Model Error]: Cannot copy out of meta tensor; no data!. Returning empty list.
 [Model Error]: Cannot copy out of meta tensor; no data!. Returning empty list.
 [Model Error]: Cannot copy out of meta tensor; no data!. Returning empty list.


 30%|██▉       | 159/537 [00:07<00:16, 22.36it/s]

 [Model Error]: Cannot copy out of meta tensor; no data!. Returning empty list.
 [Model Error]: Cannot copy out of meta tensor; no data!. Returning empty list.


KeyboardInterrupt: 

In [ ]:
import unicodedata
import re
from typing import List, Dict, Tuple, Optional
from difflib import SequenceMatcher

# --- Helper Functions ---

def _normalize_text(s: str) -> str:
    """
    Normalizes unicode, lowercases, strips, removes punctuation,
    collapses whitespace, and removes leading/trailing prepositions/articles.
    """
    if not isinstance(s, str):
        s = str(s)
    s = unicodedata.normalize("NFKC", s)
    s = s.strip().lower()
    # Remove punctuation characters
    s = "".join(ch for ch in s if not unicodedata.category(ch).startswith("P"))
    # Collapse whitespace
    s = re.sub(r"\s+", " ", s).strip()

    # Define a list of common articles and prepositions to remove
    words_to_strip = [
        'a', 'an', 'the', 'of', 'in', 'on', 'at', 'for', 'to', 'with', 'by'
    ]

    # Split the string into words
    words = s.split()

    # Remove matching words from the beginning
    while words and words[0] in words_to_strip:
        words.pop(0)

    # Remove matching words from the end
    while words and words[-1] in words_to_strip:
        words.pop(-1)

    # Rejoin the words into a final string
    s = " ".join(words)

    return s

def _opinion_sim(a: str, b: str) -> float:
    """
    Uses SequenceMatcher ratio as a lightweight similarity metric for strings.
    """
    return SequenceMatcher(None, a, b).ratio()

# --- Advanced Metrics Calculation Function ---

def calculate_metrics(
    all_preds: List[List[List[str]]],
    all_golds: List[List[List[str]]],
    normalize: bool = True,
    dedupe: bool = True,
    opinion_fuzzy_threshold: Optional[float] = None
) -> Dict[str, float]:
    """
    Computes micro precision/recall/F1 for quadruple lists with advanced matching options.
    A quadruple is [Aspect Term, Aspect Category, Sentiment Polarity, Opinion Term].
    """
    if len(all_preds) != len(all_golds):
        raise ValueError(f"Length mismatch: preds {len(all_preds)} vs golds {len(all_golds)}")

    total_tp = total_fp = total_fn = 0

    for preds, golds in zip(all_preds, all_golds):
        # Prepare lists by normalizing text if specified
        # Quadruple format: [0:Aspect, 1:Category, 2:Sentiment, 3:Opinion]
        if normalize:
            # The new normalization rule is applied here to each element
            g_list = [tuple(_normalize_text(x) for x in label) for label in golds]
            p_list = [tuple(_normalize_text(x) for x in label) for label in preds]
        else:
            g_list = [tuple(str(x) for x in label) for label in golds]
            p_list = [tuple(str(x) for x in label) for label in preds]

        # Collapse duplicate quadruples within the same example if specified
        if dedupe:
            g_list = list(dict.fromkeys(g_list))
            p_list = list(dict.fromkeys(p_list))

        matched_pred_idx = set()
        tp = 0

        # Perform a greedy one-to-one matching
        for g in g_list:
            for pj, p in enumerate(p_list):
                if pj in matched_pred_idx:
                    continue

                # CORRECTED LOGIC:
                # Aspect (0), Category (1), and Sentiment (2) must match exactly.
                non_opinion_match = (g[0] == p[0] and g[1] == p[1] and g[2] == p[2])
                if not non_opinion_match:
                    continue

                # Check for opinion match (index 3), either exact or fuzzy
                opinion_match = False
                if opinion_fuzzy_threshold is None:
                    if g[3] == p[3]: # Exact match
                        opinion_match = True
                else:
                    sim = _opinion_sim(g[3], p[3]) # Fuzzy match
                    if sim >= opinion_fuzzy_threshold:
                        opinion_match = True

                if opinion_match:
                    tp += 1
                    matched_pred_idx.add(pj)
                    break # Move to the next gold quadruple

        fp = len(p_list) - tp
        fn = len(g_list) - tp

        total_tp += tp
        total_fp += fp
        total_fn += fn

    # Calculate final metrics
    precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0
    recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    print(f"Total True Positives: {total_tp}, False Positives: {total_fp}, False Negatives: {total_fn}")

    scores = {
        'precision': precision * 100,
        'recall': recall * 100,
        'f1': f1 * 100
    }
    return scores

def main():
    """
    Main function to load results from a JSON file and compute the F1 score.
    """
    # IMPORTANT: Update this path to point to your results file
    file_path = "/content/Rest16_evaluation_0shot_results_Meta-Llama-3.1-8B-Instruct.json"
    all_predictions = []
    all_ground_truths = []

    print(f"Loading results from '{file_path}'...")

    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            results_data = json.load(f)
    except FileNotFoundError:
        print(f"Error: The file '{file_path}' was not found. Please check the path.")
        return
    except json.JSONDecodeError:
        print(f"Error: The file '{file_path}' is not a valid JSON file.")
        return

    # Extract the predictions and ground truth labels from the loaded data
    for item in results_data:
        all_predictions.append(item.get("prediction", []))
        all_ground_truths.append(item.get("ground_truth", []))

    if not all_predictions or not all_ground_truths:
        print("Error: Could not find 'prediction' or 'ground_truth' keys in the JSON file.")
        return

    print(f"Successfully loaded {len(all_predictions)} samples.")

    # --- Calculate and Report Final Metrics ---
    print("\nCalculating scores with advanced metrics...")

    # You can experiment with fuzzy matching by setting a threshold, e.g., opinion_fuzzy_threshold=0.85
    final_metrics = calculate_metrics(
        all_preds=all_predictions,
        all_golds=all_ground_truths,
        normalize=True,
        dedupe=True,
        opinion_fuzzy_threshold=None # Set to a float like 0.85 for fuzzy matching
    )

    print("\n\n========== FINAL METRICS ==========")
    print(f"  Precision: {final_metrics['precision']:.2f}%")
    print(f"  Recall:    {final_metrics['recall']:.2f}%")
    print(f"  F1-Score:  {final_metrics['f1']:.2f}%")
    print("===================================")


if __name__ == "__main__":
    main()

Loading results from '/content/Rest16_evaluation_0shot_results_Meta-Llama-3.1-8B-Instruct.json'...
Successfully loaded 544 samples.

Calculating scores with advanced metrics...
Total True Positives: 158, False Positives: 790, False Negatives: 641


========== FINAL METRICS ==========
  Precision: 16.67%
  Recall:    19.77%
  F1-Score:  18.09%


# **llama 3.1 70 B**

In [ ]:
import torch
import spacy
import ast
import re
import json
import unicodedata
import numpy as np
from tqdm import tqdm
from typing import List, Dict, Tuple, Optional
from difflib import SequenceMatcher
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from sklearn.metrics.pairwise import cosine_similarity

# --- Main Configuration ---
#MODEL_ID = "meta-llama/Meta-Llama-3.1-8B-Instruct"
MODEL_ID = "unsloth/Meta-Llama-3.1-70B-Instruct-bnb-4bit"

TRAIN_DATA_PATH = "train.txt"
TEST_DATA_PATH = "test.txt"

# --- Helper Functions ---
def read_line_examples_from_file(data_path: str) -> Tuple[List[str], List[list]]:
    """Reads data from file, each line is: sent####labels"""
    sents, labels = [], []
    with open(data_path, 'r', encoding='UTF-8') as fp:
        for line in fp:
            line = line.strip()
            if line != '' and '####' in line:
                words, tuples_str = line.split('####', 1)
                try:
                    tuples = ast.literal_eval(tuples_str)
                    sents.append(words)
                    labels.append(tuples)
                except (ValueError, SyntaxError):
                    print(f"Skipping malformed line: {line}")
    return sents, labels

# --- Advanced Metrics Calculation Section ---
def _normalize_text(s: str) -> str:
    """ Normalizes unicode, lowercases, strips, removes punctuation, collapses whitespace. """
    if not isinstance(s, str):
        s = str(s)
    s = unicodedata.normalize("NFKC", s)
    s = s.strip().lower()
    s = "".join(ch for ch in s if not unicodedata.category(ch).startswith("P"))
    s = re.sub(r"\s+", " ", s).strip()
    words_to_strip = ['a', 'an', 'the', 'of', 'in', 'on', 'at', 'for', 'to', 'with', 'by']
    words = s.split()
    while words and words[0] in words_to_strip:
        words.pop(0)
    while words and words[-1] in words_to_strip:
        words.pop(-1)
    s = " ".join(words)
    return s

def _opinion_sim(a: str, b: str) -> float:
    """ Uses SequenceMatcher ratio as a lightweight similarity metric for strings. """
    return SequenceMatcher(None, a, b).ratio()

def calculate_metrics(
    all_preds: List[List[List[str]]],
    all_golds: List[List[List[str]]],
    normalize: bool = True,
    dedupe: bool = True,
    opinion_fuzzy_threshold: Optional[float] = None
) -> Dict[str, float]:
    """ Computes micro precision/recall/F1 for quadruple lists with advanced matching options. """
    if len(all_preds) != len(all_golds):
        raise ValueError(f"Length mismatch: preds {len(all_preds)} vs golds {len(all_golds)}")

    total_tp = total_fp = total_fn = 0
    for preds, golds in zip(all_preds, all_golds):
        if normalize:
            g_list = [tuple(_normalize_text(x) for x in label) for label in golds]
            p_list = [tuple(_normalize_text(x) for x in label) for label in preds]
        else:
            g_list = [tuple(str(x) for x in label) for label in golds]
            p_list = [tuple(str(x) for x in label) for label in preds]

        if dedupe:
            g_list = list(dict.fromkeys(g_list))
            p_list = list(dict.fromkeys(p_list))

        matched_pred_idx = set()
        tp = 0
        for g in g_list:
            for pj, p in enumerate(p_list):
                if pj in matched_pred_idx:
                    continue

                non_opinion_match = (g[0] == p[0] and g[1] == p[1] and g[2] == p[2])
                if not non_opinion_match:
                    continue

                opinion_match = False
                if opinion_fuzzy_threshold is None:
                    if g[3] == p[3]:
                        opinion_match = True
                else:
                    sim = _opinion_sim(g[3], p[3])
                    if sim >= opinion_fuzzy_threshold:
                        opinion_match = True

                if opinion_match:
                    tp += 1
                    matched_pred_idx.add(pj)
                    break

        fp = len(p_list) - tp
        fn = len(g_list) - tp
        total_tp += tp
        total_fp += fp
        total_fn += fn

    print(f"\nCalculating scores with advanced metrics...")
    print(f"Total True Positives: {total_tp}, False Positives: {total_fp}, False Negatives: {total_fn}")
    precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0
    recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    return {'precision': precision * 100, 'recall': recall * 100, 'f1': f1 * 100}

class Llama_ASQPExtractor:
    """An extractor class to evaluate Llama 3.1 models via transformers."""
    def __init__(self, model_id: str, token: str = None):
        print(f"Initializing Llama Extractor for model: {model_id}")
        # quantization_config = BitsAndBytesConfig(
        #     load_in_4bit=True,
        #     bnb_4bit_quant_type="nf4",
        #     bnb_4bit_compute_dtype=torch.bfloat16,

        # )

        print("Loading tokenizer...")
        self.tokenizer = AutoTokenizer.from_pretrained(
            model_id,
            token=token,
            trust_remote_code=True # <-- ADDED: Required for Unsloth
        )

        # ADDED: Good practice for generation stability
        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

        print("Loading model (this may take a while)...")
        self.model = AutoModelForCausalLM.from_pretrained(
            model_id,
            torch_dtype=torch.bfloat16,
            device_map="auto",
            # quantization_config=quantization_config,
            token=token,
            trust_remote_code=True # <-- ADDED: Required for Unsloth

        )

        self.nlp_sm = spacy.load("en_core_web_sm")
        print("Model and tools initialized successfully!")

    def _create_prompt(self, instruction: str, query: str, dependency_tree: str, few_shot_examples: List[Dict]) -> str:
        """Formats the prompt into the Llama 3.1 Instruct chat format."""
        system_prompt = instruction

        few_shot_text = ""
        if few_shot_examples:
            few_shot_text = "### Examples\nHere are some examples to guide you:\n"
            for example in few_shot_examples:
                few_shot_text += f"\nSentence: {example['sentence']}\nLabels: {str(example['labels'])}\n"
            few_shot_text += "\n### Now, based on instruction and these examples, perform the task for the following sentence:\n"
        else:
            few_shot_text = "### Now, based on the instruction, perform the task for the following sentence:\n"

        user_prompt = f"{few_shot_text}Query: {query}\n"
        user_prompt += f"Dependency Tree: {dependency_tree}\n"
        user_prompt += "LABELS:"

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ]

        formatted_prompt = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        return formatted_prompt

    def _parse_output(self, output_text: str) -> List[list]:
        """Robustly parse the model's string output into a Python list of lists."""
        # Clean up the output text
        output_text = output_text.strip()

        # Find the start of the list
        start = output_text.find('[')
        if start == -1:
            return []

        # Use a stack to find the matching closing bracket
        stack = 0
        end = -1
        for i in range(start, len(output_text)):
            ch = output_text[i]
            if ch == '[':
                stack += 1
            elif ch == ']':
                stack -= 1
                if stack == 0:
                    end = i
                    break
        if end == -1:
            return []

        list_str = output_text[start:end+1]

        # Replace single quotes with double quotes for proper JSON parsing
        # But be careful not to replace quotes inside words (contractions)
        list_str = re.sub(r"(?<![a-zA-Z])'(?![a-zA-Z])", '"', list_str)

        # Try multiple parsing strategies
        try:
            # Strategy 1: Use ast.literal_eval (handles Python syntax)
            parsed_list = ast.literal_eval(list_str)
            if isinstance(parsed_list, list) and all(isinstance(i, (list, tuple)) and len(i) == 4 for i in parsed_list):
                # Convert tuples to lists and ensure all elements are strings
                return [[str(elem) for elem in item] for item in parsed_list]
        except Exception:
            pass

        try:
            # Strategy 2: Use json.loads (handles JSON syntax)
            parsed_list = json.loads(list_str)
            if isinstance(parsed_list, list) and all(isinstance(i, list) and len(i) == 4 for i in parsed_list):
                # Ensure all elements are strings
                return [[str(elem) for elem in item] for item in parsed_list]
        except Exception:
            pass

        # Strategy 3: Manual regex-based parsing as fallback
        try:
            # Extract all quadruples using regex
            quadruple_pattern = r'\[(.*?)\]'
            matches = re.findall(quadruple_pattern, list_str)

            result = []
            for match in matches:
                # Skip the outer list match
                if match == list_str[1:-1]:
                    continue

                # Split by comma but respect quoted strings
                elements = re.findall(r'["\']([^"\']*)["\']|([^,\[\]]+)', match)
                elements = [e[0] if e[0] else e[1].strip() for e in elements if e[0] or e[1].strip()]

                if len(elements) == 4:
                    result.append([str(elem) for elem in elements])

            if result:
                return result
        except Exception:
            pass

        return []

    def extract_quadruples(self, instruction: str, sentence: str, few_shot_examples: List[Dict]) -> List[list]:
        """Gets a prediction from the Llama model for a given sentence."""
        doc = self.nlp_sm(sentence)
        dep_lines = [f"{token.text}-{token.head.text}->{token.dep_}" for token in doc]
        dependency_tree = "\n".join(dep_lines)

        prompt = self._create_prompt(instruction, sentence, dependency_tree, few_shot_examples)

        inputs = self.tokenizer(prompt, return_tensors="pt").to("cuda")

        terminators = [
            self.tokenizer.eos_token_id,
            self.tokenizer.convert_tokens_to_ids("<|eot_id|>")
        ]
        try:
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=512,
                eos_token_id=terminators,
                do_sample=False,
                pad_token_id=self.tokenizer.eos_token_id
            )
            response_text = self.tokenizer.decode(outputs[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)
            parsed = self._parse_output(response_text)
            # Only show debug if parsing failed AND there was actual content
            if not parsed and response_text.strip() and not response_text.strip().startswith('[]'):
                # Check if it looks like it should have been parseable
                if '[' in response_text and ']' in response_text:
                    print(f"⚠ WARNING: Failed to parse output with brackets: \n{response_text[:150]}...")
            return parsed
        except Exception as e:
            print(f" [Model Error]: {e}. Returning empty list.")
            return []

if __name__ == "__main__":
    NUM_FEW_SHOTS = 10
    hf_token = None
    extractor = Llama_ASQPExtractor(model_id=MODEL_ID, token=hf_token)

    print("\nLoading datasets and building the few-shot retriever...")
    train_sents, train_labels = read_line_examples_from_file(TRAIN_DATA_PATH)
    test_sents, test_labels = read_line_examples_from_file(TEST_DATA_PATH)

    if NUM_FEW_SHOTS > 0:
        nlp_lg = spacy.load("en_core_web_lg")
        print("Building few-shot retriever index (this may take a moment)...")
        train_embeddings = np.array([nlp_lg(sent).vector for sent in tqdm(train_sents)])

    def retrieve_few_shots(query_sentence: str, num_examples: int = 1) -> List[Dict]:
        if num_examples <= 0:
            return []
        query_embedding = nlp_lg(query_sentence).vector.reshape(1, -1)
        similarities = cosine_similarity(query_embedding, train_embeddings).flatten()
        top_indices = np.argsort(similarities)[-num_examples:][::-1]
        return [{"sentence": train_sents[idx], "labels": train_labels[idx]} for idx in top_indices]

    instruction_prompt = """
ROLE:
You are an expert ASQP (aspect sentiment quadruple prediction) annotation bot. Your processing must be precise, systematic, and strictly follow all rules.

TASK DEFINITION:
Your sole task is to extract all sentiment quadruples from a user-provided sentence and its dependency tree.
Your response MUST be a valid Python list of lists, with no other text or explanations.
The output format for each quadruple is a list of four strings:
[<Aspect Term>, <Aspect Category>, <Sentiment Polarity>, <Opinion Term>]

ELEMENT DEFINITIONS:
1.  Aspect Term: The specific noun or phrase being reviewed.
    - It MUST be a term from the restaurant domain (e.g., "food", "service", "waiter").
    - Use the exact string "NULL" if the opinion is general or the aspect is implicit.
2.  Aspect Category: The general class for the Aspect Term.
    - You MUST choose from this predefined list only:
        - `ambience general`
        - `drinks prices`
        - `drinks quality`
        - `drinks style_options`
        - `food prices`
        - `food quality`
        - `food style_options`
        - `location general`
        - `restaurant general`
        - `restaurant miscellaneous`
        - `restaurant prices`
        - `service general`
3.  Sentiment: The polarity of the opinion.
    - It MUST be one of three strings: "positive", "negative", or "neutral".
4.  Opinion Term: The exact word(s) from the sentence expressing the sentiment.

EXTRACTION RULES:
1.  Concise Spans: The Opinion Term must be the most concise phrase that still captures the full sentiment. Do not include unnecessary words.
2.  Extract All Opinions: You must extract a separate quadruple for every distinct opinion expressed in the sentence.
3.  Dependency Tree Usage: To ensure accuracy, you must use the dependency tree to find the correct relationship between an opinion and an aspect. Follow the grammatical connections in the tree to link an Opinion Term back to the specific Aspect Term it modifies. This is the most reliable way to create a correct quadruple. If no such link exists, the Aspect Term is "NULL".
4.  Implied Sentiment: Analyze comparative and conditional phrases to determine the implied sentiment.

STEP BY STEP PROCESS:
To ensure accuracy, think step-by-step before generating the final list:
1.  First, identify all opinion-expressing words/phrases in the sentence.
2.  Second, for each opinion, identify its specific aspect phrase. If none exist, that aspect is "NULL".
3.  Third, determine the sentiment polarity (positive, negative, neutral) and the correct aspect category from the provided list.
4.  Finally, construct the list of all quadruples according to all rules above.

FINAL OUTPUT INSTRUCTION:
Your final output MUST ONLY be a valid Python list of lists.
- Do not include explanations, reasoning, or markdown formatting.
- Do not include extra text before or after the list.
- Each inner list must contain exactly 4 strings.
- Use double quotes for strings, not single quotes.
"""

    all_predictions = []
    all_ground_truths = test_labels

    print(f"\n--- Starting Evaluation on {len(test_sents)} Test Samples ---")
    for i, sentence in enumerate(tqdm(test_sents)):
        few_shot_examples = retrieve_few_shots(sentence, num_examples=NUM_FEW_SHOTS) if NUM_FEW_SHOTS > 0 else []
        predicted_quads = extractor.extract_quadruples(instruction_prompt, sentence, few_shot_examples)
        all_predictions.append(predicted_quads)

        if (i + 1) % 10 == 0:
            print(f"\n----- Sample {i+1}/{len(test_sents)} -----")
            print(f" Sentence: {sentence}")
            print(f" Ground Truth: {all_ground_truths[i]}")
            print(f" Prediction: {predicted_quads}")
            print("-" * 25)

    final_metrics = calculate_metrics(all_predictions, all_ground_truths)
    print("\n\n========== EVALUATION COMPLETE ==========")
    print(f" Model: {MODEL_ID}")
    print(f" Few-shots: {NUM_FEW_SHOTS}")
    print(f" Precision: {final_metrics['precision']:.2f}%")
    print(f" Recall: {final_metrics['recall']:.2f}%")
    print(f" F1-Score: {final_metrics['f1']:.2f}%")
    print("=======================================")

    model_name_safe = MODEL_ID.split('/')[-1]
    output_filename = f"Rest16_evaluation_{NUM_FEW_SHOTS}shot_results_{model_name_safe}.json"
    results_to_save = [
        {"sentence": s, "ground_truth": gt, "prediction": p}
        for s, gt, p in zip(test_sents, all_ground_truths, all_predictions)
    ]
    with open(output_filename, "w", encoding="utf-8") as f:
        json.dump(results_to_save, f, indent=4)
    print(f"\nDetailed results saved to '{output_filename}'")

Initializing Llama Extractor for model: unsloth/Meta-Llama-3.1-70B-Instruct-bnb-4bit
Loading tokenizer...
Loading model (this may take a while)...


Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

Model and tools initialized successfully!

Loading datasets and building the few-shot retriever...
Building few-shot retriever index (this may take a moment)...


100%|██████████| 1264/1264 [00:09<00:00, 139.20it/s]



--- Starting Evaluation on 544 Test Samples ---


...


----- Sample 530/544 -----
 Sentence: It 's just the right size for the menu .
 Ground Truth: [['NULL', 'restaurant miscellaneous', 'positive', 'the right size']]
 Prediction: [['NULL', 'restaurant miscellaneous', 'positive', 'just the right size']]
-------------------------


 99%|█████████▉| 540/544 [49:46<00:27,  6.90s/it]


----- Sample 540/544 -----
 Sentence: In fact , many want to return a second time during their visit .
 Ground Truth: [['NULL', 'restaurant general', 'positive', 'return a second time']]
 Prediction: [['NULL', 'restaurant general', 'positive', 'want to return']]
-------------------------


100%|██████████| 544/544 [49:59<00:00,  5.51s/it]


Calculating scores with advanced metrics...
Total True Positives: 427, False Positives: 504, False Negatives: 372


========== EVALUATION COMPLETE ==========
 Model: unsloth/Meta-Llama-3.1-70B-Instruct-bnb-4bit
 Few-shots: 10
 Precision: 45.86%
 Recall: 53.44%
 F1-Score: 49.36%

Detailed results saved to 'Rest16_evaluation_10shot_results_Meta-Llama-3.1-70B-Instruct-bnb-4bit.json'


In [ ]:
import unicodedata
import re
from typing import List, Dict, Tuple, Optional
from difflib import SequenceMatcher

# --- Helper Functions ---

def _normalize_text(s: str) -> str:
    """
    Normalizes unicode, lowercases, strips, removes punctuation,
    collapses whitespace, and removes leading/trailing prepositions/articles.
    """
    if not isinstance(s, str):
        s = str(s)
    s = unicodedata.normalize("NFKC", s)
    s = s.strip().lower()
    # Remove punctuation characters
    s = "".join(ch for ch in s if not unicodedata.category(ch).startswith("P"))
    # Collapse whitespace
    s = re.sub(r"\s+", " ", s).strip()

    # Define a list of common articles and prepositions to remove
    words_to_strip = [
        'a', 'an', 'the', 'of', 'in', 'on', 'at', 'for', 'to', 'with', 'by'
    ]

    # Split the string into words
    words = s.split()

    # Remove matching words from the beginning
    while words and words[0] in words_to_strip:
        words.pop(0)

    # Remove matching words from the end
    while words and words[-1] in words_to_strip:
        words.pop(-1)

    # Rejoin the words into a final string
    s = " ".join(words)

    return s

def _opinion_sim(a: str, b: str) -> float:
    """
    Uses SequenceMatcher ratio as a lightweight similarity metric for strings.
    """
    return SequenceMatcher(None, a, b).ratio()

# --- Advanced Metrics Calculation Function ---

def calculate_metrics(
    all_preds: List[List[List[str]]],
    all_golds: List[List[List[str]]],
    normalize: bool = True,
    dedupe: bool = True,
    opinion_fuzzy_threshold: Optional[float] = None
) -> Dict[str, float]:
    """
    Computes micro precision/recall/F1 for quadruple lists with advanced matching options.
    A quadruple is [Aspect Term, Aspect Category, Sentiment Polarity, Opinion Term].
    """
    if len(all_preds) != len(all_golds):
        raise ValueError(f"Length mismatch: preds {len(all_preds)} vs golds {len(all_golds)}")

    total_tp = total_fp = total_fn = 0

    for preds, golds in zip(all_preds, all_golds):
        # Prepare lists by normalizing text if specified
        # Quadruple format: [0:Aspect, 1:Category, 2:Sentiment, 3:Opinion]
        if normalize:
            # The new normalization rule is applied here to each element
            g_list = [tuple(_normalize_text(x) for x in label) for label in golds]
            p_list = [tuple(_normalize_text(x) for x in label) for label in preds]
        else:
            g_list = [tuple(str(x) for x in label) for label in golds]
            p_list = [tuple(str(x) for x in label) for label in preds]

        # Collapse duplicate quadruples within the same example if specified
        if dedupe:
            g_list = list(dict.fromkeys(g_list))
            p_list = list(dict.fromkeys(p_list))

        matched_pred_idx = set()
        tp = 0

        # Perform a greedy one-to-one matching
        for g in g_list:
            for pj, p in enumerate(p_list):
                if pj in matched_pred_idx:
                    continue

                # CORRECTED LOGIC:
                # Aspect (0), Category (1), and Sentiment (2) must match exactly.
                non_opinion_match = (g[0] == p[0] and g[1] == p[1] and g[2] == p[2])
                if not non_opinion_match:
                    continue

                # Check for opinion match (index 3), either exact or fuzzy
                opinion_match = False
                if opinion_fuzzy_threshold is None:
                    if g[3] == p[3]: # Exact match
                        opinion_match = True
                else:
                    sim = _opinion_sim(g[3], p[3]) # Fuzzy match
                    if sim >= opinion_fuzzy_threshold:
                        opinion_match = True

                if opinion_match:
                    tp += 1
                    matched_pred_idx.add(pj)
                    break # Move to the next gold quadruple

        fp = len(p_list) - tp
        fn = len(g_list) - tp

        total_tp += tp
        total_fp += fp
        total_fn += fn

    # Calculate final metrics
    precision = total_tp / (total_tp + total_fp) if (total_tp + total_fp) > 0 else 0
    recall = total_tp / (total_tp + total_fn) if (total_tp + total_fn) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    print(f"Total True Positives: {total_tp}, False Positives: {total_fp}, False Negatives: {total_fn}")

    scores = {
        'precision': precision * 100,
        'recall': recall * 100,
        'f1': f1 * 100
    }
    return scores

def main():
    """
    Main function to load results from a JSON file and compute the F1 score.
    """
    # IMPORTANT: Update this path to point to your results file
    file_path = "/content/Rest16_evaluation_10shot_results_Meta-Llama-3.1-70B-Instruct-bnb-4bit.json"
    all_predictions = []
    all_ground_truths = []

    print(f"Loading results from '{file_path}'...")

    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            results_data = json.load(f)
    except FileNotFoundError:
        print(f"Error: The file '{file_path}' was not found. Please check the path.")
        return
    except json.JSONDecodeError:
        print(f"Error: The file '{file_path}' is not a valid JSON file.")
        return

    # Extract the predictions and ground truth labels from the loaded data
    for item in results_data:
        all_predictions.append(item.get("prediction", []))
        all_ground_truths.append(item.get("ground_truth", []))

    if not all_predictions or not all_ground_truths:
        print("Error: Could not find 'prediction' or 'ground_truth' keys in the JSON file.")
        return

    print(f"Successfully loaded {len(all_predictions)} samples.")

    # --- Calculate and Report Final Metrics ---
    print("\nCalculating scores with advanced metrics...")

    # You can experiment with fuzzy matching by setting a threshold, e.g., opinion_fuzzy_threshold=0.85
    final_metrics = calculate_metrics(
        all_preds=all_predictions,
        all_golds=all_ground_truths,
        normalize=True,
        dedupe=True,
        opinion_fuzzy_threshold=None # Set to a float like 0.85 for fuzzy matching
    )

    print("\n\n========== FINAL METRICS ==========")
    print(f"  Precision: {final_metrics['precision']:.2f}%")
    print(f"  Recall:    {final_metrics['recall']:.2f}%")
    print(f"  F1-Score:  {final_metrics['f1']:.2f}%")
    print("===================================")


if __name__ == "__main__":
    main()

Loading results from '/content/Rest16_evaluation_10shot_results_Meta-Llama-3.1-70B-Instruct-bnb-4bit.json'...
Successfully loaded 544 samples.

Calculating scores with advanced metrics...
Total True Positives: 427, False Positives: 504, False Negatives: 372


========== FINAL METRICS ==========
  Precision: 45.86%
  Recall:    53.44%
  F1-Score:  49.36%
